<a href="https://colab.research.google.com/github/YieldShock13/test-edv/blob/main/Copy_of_Untitled270.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# US TREASURY PAR CURVE -> INTERPOLATION -> SPOT CURVE
# Textbook Sequential Bootstrap
# Robust Google Colab + Interactive Plotly
# ============================================================

import requests
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from scipy.interpolate import PchipInterpolator
from datetime import datetime

import plotly.graph_objects as go
import plotly.io as pio

# Explicit Colab renderer
pio.renderers.default = "colab"


# ------------------------------------------------------------
# 1. PULL LATEST CURVE FROM OFFICIAL TREASURY API
# ------------------------------------------------------------

year = datetime.now().year

url = (
    "https://home.treasury.gov/resource-center/data-chart-center/"
    "interest-rates/pages/xml"
)

params = {
    "data": "daily_treasury_yield_curve",
    "field_tdr_date_value": year
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

root = ET.fromstring(response.content)

ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "m": "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata"
}

rows = []

for entry in root.findall("atom:entry", ns):

    props = entry.find("atom:content/m:properties", ns)

    if props is None:
        continue

    row = {}

    for child in props:
        field = child.tag.split("}")[-1]
        row[field] = child.text

    rows.append(row)

raw = pd.DataFrame(rows)

if raw.empty:
    raise RuntimeError("Treasury API returned no data.")

raw["NEW_DATE"] = pd.to_datetime(raw["NEW_DATE"])
raw = raw.sort_values("NEW_DATE")

latest = raw.iloc[-1]
valuation_date = latest["NEW_DATE"]

print("Curve date:", valuation_date.date())


# ------------------------------------------------------------
# 2. REQUIRED TREASURY PAR NODES
# ------------------------------------------------------------

nodes = {
    "BC_6MONTH": 0.5,
    "BC_1YEAR": 1.0,
    "BC_2YEAR": 2.0,
    "BC_3YEAR": 3.0,
    "BC_5YEAR": 5.0,
    "BC_7YEAR": 7.0,
    "BC_10YEAR": 10.0,
    "BC_20YEAR": 20.0,
    "BC_30YEAR": 30.0
}

# IMPORTANT:
# Do NOT silently continue if Treasury has omitted a required node.

missing = []

for field in nodes:

    if (
        field not in latest.index
        or pd.isna(latest[field])
        or latest[field] == ""
    ):
        missing.append(field)

if missing:
    raise RuntimeError(
        "Required Treasury par-curve nodes are missing: "
        + ", ".join(missing)
    )


par_nodes = pd.DataFrame({
    "Maturity": list(nodes.values()),
    "Par_Yield": [
        float(latest[field])
        for field in nodes
    ]
})

print("\nOfficial Treasury par nodes:")
print(par_nodes.to_string(index=False))


# ------------------------------------------------------------
# 3. CREATE EXACT SEMIANNUAL GRID
# ------------------------------------------------------------

# Do NOT use np.arange for the maturity grid.
#
# Integer construction guarantees:
# 0.5, 1.0, 1.5 ... 29.5, 30.0
#
# Therefore 30Y cannot disappear or duplicate due to
# floating-point accumulation.

max_maturity = float(par_nodes["Maturity"].max())

n_periods = int(round(max_maturity * 2))

maturities = (
    np.arange(1, n_periods + 1, dtype=int) / 2.0
)

# Explicit validation
assert maturities[0] == 0.5
assert maturities[-1] == max_maturity
assert len(np.unique(maturities)) == len(maturities)

print(
    f"\nSemiannual grid validated: "
    f"{maturities[0]:.1f}Y to {maturities[-1]:.1f}Y "
    f"({len(maturities)} nodes)"
)


# ------------------------------------------------------------
# 4. INTERPOLATE PAR YIELDS
# ------------------------------------------------------------

interpolator = PchipInterpolator(
    par_nodes["Maturity"].to_numpy(),
    par_nodes["Par_Yield"].to_numpy()
)

par_yields = interpolator(maturities)

semiannual_curve = pd.DataFrame({
    "Maturity": maturities,
    "Par_Yield": par_yields
})


# ------------------------------------------------------------
# 5. TEXTBOOK SEQUENTIAL BOOTSTRAP
# ------------------------------------------------------------

discount_factors = []
spot_rates = []

for i, row in semiannual_curve.iterrows():

    T = row["Maturity"]

    par_rate = row["Par_Yield"] / 100.0

    # Semiannual coupon per $100 face
    coupon = 100.0 * par_rate / 2.0

    # All previously solved discount factors
    if i == 0:
        pv_previous_coupons = 0.0
    else:
        pv_previous_coupons = (
            coupon * np.sum(discount_factors)
        )

    # 100 = C*DF1 + C*DF2 + ... + (100+C)*DFn

    df_T = (
        100.0 - pv_previous_coupons
    ) / (
        100.0 + coupon
    )

    if df_T <= 0:
        raise RuntimeError(
            f"Invalid discount factor at {T}Y: {df_T}"
        )

    discount_factors.append(df_T)

    # Semiannual-compounded spot rate

    spot = 2.0 * (
        df_T ** (-1.0 / (2.0 * T)) - 1.0
    )

    spot_rates.append(spot * 100.0)


semiannual_curve["Discount_Factor"] = discount_factors
semiannual_curve["Spot_Yield"] = spot_rates


# ------------------------------------------------------------
# 6. FINAL VALIDATION
# ------------------------------------------------------------

if semiannual_curve["Maturity"].iloc[-1] != 30.0:
    raise RuntimeError("30Y maturity missing from final curve.")

if len(semiannual_curve) != 60:
    raise RuntimeError(
        f"Expected 60 semiannual nodes through 30Y; "
        f"received {len(semiannual_curve)}."
    )

if semiannual_curve.isna().any().any():
    raise RuntimeError("NaN detected in final spot curve.")

print("\nFinal curve validated successfully.")


# ------------------------------------------------------------
# 7. SAVE DATA
# ------------------------------------------------------------

par_nodes.to_csv(
    "/content/treasury_official_par_nodes.csv",
    index=False
)

semiannual_curve.to_csv(
    "/content/treasury_bootstrapped_spot_curve.csv",
    index=False
)


# ------------------------------------------------------------
# 8. INTERACTIVE PLOTLY CHART
# ------------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=semiannual_curve["Maturity"],
        y=semiannual_curve["Par_Yield"],
        mode="lines",
        name="Interpolated Par Curve",
        hovertemplate=(
            "Maturity: %{x:.1f}Y<br>"
            "Par Yield: %{y:.3f}%"
            "<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=semiannual_curve["Maturity"],
        y=semiannual_curve["Spot_Yield"],
        mode="lines",
        name="Bootstrapped Spot Curve",
        hovertemplate=(
            "Maturity: %{x:.1f}Y<br>"
            "Spot Yield: %{y:.3f}%"
            "<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Scatter(
        x=par_nodes["Maturity"],
        y=par_nodes["Par_Yield"],
        mode="markers",
        name="Official Treasury Nodes",
        marker=dict(size=8),
        hovertemplate=(
            "Official Treasury Node<br>"
            "Maturity: %{x:.1f}Y<br>"
            "Par Yield: %{y:.3f}%"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title=dict(
        text=(
            "U.S. Treasury Par and Bootstrapped Spot Curve"
            f"<br><sup>{valuation_date.date()}</sup>"
        ),
        x=0.5,
        xanchor="center"
    ),

    font=dict(
        family="Arial, sans-serif",
        size=14
    ),

    xaxis=dict(
        title="Maturity (Years)",
        showgrid=True,
        gridcolor="rgba(128,128,128,0.15)",
        zeroline=False
    ),

    yaxis=dict(
        title="Yield (%)",
        showgrid=True,
        gridcolor="rgba(128,128,128,0.15)",
        zeroline=False
    ),

    template="plotly_white",

    hovermode="x unified",

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),

    width=1100,
    height=650,

    margin=dict(
        l=80,
        r=40,
        t=120,
        b=70
    )
)

fig.show(renderer="colab")


# ------------------------------------------------------------
# 9. FULL NUMERICAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 78)
print(f"U.S. TREASURY CURVE — {valuation_date.date()}")
print("=" * 78)

# Official Treasury nodes
print("\nOFFICIAL TREASURY PAR YIELDS")
print("-" * 45)

official_output = par_nodes.copy()

official_output["Maturity"] = official_output["Maturity"].map(
    lambda x: f"{x:.1f}Y"
)

official_output["Par_Yield"] = official_output["Par_Yield"].map(
    lambda x: f"{x:.4f}%"
)

print(
    official_output.to_string(
        index=False,
        header=["Maturity", "Par Yield"]
    )
)


# Full interpolated + bootstrapped curve
print("\nINTERPOLATED PAR + BOOTSTRAPPED SPOT CURVE")
print("-" * 78)

curve_output = semiannual_curve.copy()

curve_output["Maturity"] = curve_output["Maturity"].map(
    lambda x: f"{x:.1f}Y"
)

curve_output["Par_Yield"] = curve_output["Par_Yield"].map(
    lambda x: f"{x:.4f}%"
)

curve_output["Spot_Yield"] = curve_output["Spot_Yield"].map(
    lambda x: f"{x:.4f}%"
)

curve_output["Discount_Factor"] = curve_output["Discount_Factor"].map(
    lambda x: f"{x:.8f}"
)

print(
    curve_output[
        [
            "Maturity",
            "Par_Yield",
            "Spot_Yield",
            "Discount_Factor"
        ]
    ].to_string(
        index=False,
        header=[
            "Maturity",
            "Par Yield",
            "Spot Yield",
            "Discount Factor"
        ]
    )
)


# ------------------------------------------------------------
# 10. SAVED FILES
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("FILES SAVED")
print("=" * 78)

print("/content/treasury_official_par_nodes.csv")
print("/content/treasury_bootstrapped_spot_curve.csv")

Curve date: 2026-09-17

Official Treasury par nodes:
 Maturity  Par_Yield
      0.5       4.20
      1.0       4.40
      2.0       4.67
      3.0       4.75
      5.0       4.78
      7.0       4.86
     10.0       4.94
     20.0       5.32
     30.0       5.29

Semiannual grid validated: 0.5Y to 30.0Y (60 nodes)

Final curve validated successfully.



U.S. TREASURY CURVE — 2026-09-17

OFFICIAL TREASURY PAR YIELDS
---------------------------------------------
Maturity Par Yield
    0.5Y   4.2000%
    1.0Y   4.4000%
    2.0Y   4.6700%
    3.0Y   4.7500%
    5.0Y   4.7800%
    7.0Y   4.8600%
   10.0Y   4.9400%
   20.0Y   5.3200%
   30.0Y   5.2900%

INTERPOLATED PAR + BOOTSTRAPPED SPOT CURVE
------------------------------------------------------------------------------
Maturity Par Yield Spot Yield Discount Factor
    0.5Y   4.2000%    4.2000%      0.97943193
    1.0Y   4.4000%    4.4022%      0.95738992
    1.5Y   4.5608%    4.5660%      0.93452247
    2.0Y   4.6700%    4.6779%      0.91166669
    2.5Y   4.7220%    4.7309%      0.88967757
    3.0Y   4.7500%    4.7592%      0.86839917
    3.5Y   4.7603%    4.7690%      0.84793076
    4.0Y   4.7664%    4.7745%      0.82800459
    4.5Y   4.7717%    4.7795%      0.80852097
    5.0Y   4.7800%    4.7880%      0.78932464
    5.5Y   4.7956%    4.8049%      0.77017014
    6.0Y   4.8173%    4.8

In [ ]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime
from scipy.interpolate import PchipInterpolator
from dateutil.relativedelta import relativedelta

# ============================================================
# SETTINGS
# ============================================================

VALUATION_DATE = pd.Timestamp("2026-09-17")
FACE = 100.0

CURVE_FILE = "/content/treasury_bootstrapped_spot_curve.csv"

CUSIPS = [
    "91282CLL3",
    "9128284N7",
    "912828YS3",
    "912828Z94",
    "91282CCB5",
    "91282CDY4",
    "91282CGM7",
    "91282CJY8",
    "91282CNS6",
    "912810QF8",
    "912810RL4",
    "912810SG4",
    "912810SM1",
    "912810TP3",
    "912810UG1",
]

# ============================================================
# 1. LOAD OUR BOOTSTRAPPED SPOT CURVE
# ============================================================

curve = pd.read_csv(CURVE_FILE)

# Flexible column detection
def find_col(df, candidates):
    for candidate in candidates:
        for col in df.columns:
            if candidate.lower() in col.lower():
                return col
    raise RuntimeError(
        f"Could not identify required column.\n"
        f"Available columns: {list(df.columns)}"
    )

maturity_col = find_col(curve, ["maturity"])
spot_col = find_col(curve, ["spot"])

curve = curve[[maturity_col, spot_col]].copy()
curve.columns = ["Maturity", "Spot"]

curve["Maturity"] = pd.to_numeric(curve["Maturity"], errors="coerce")
curve["Spot"] = pd.to_numeric(curve["Spot"], errors="coerce")

curve = curve.dropna().sort_values("Maturity")

# Handle percent-form versus decimal-form spot rates
if curve["Spot"].median() > 1:
    curve["Spot"] /= 100.0

if abs(curve["Maturity"].iloc[0] - 0.5) > 1e-8:
    raise RuntimeError("Expected first spot-curve node at 0.5 years.")

if abs(curve["Maturity"].iloc[-1] - 30.0) > 1e-8:
    raise RuntimeError("Expected final spot-curve node at 30 years.")

spot_interp = PchipInterpolator(
    curve["Maturity"].values,
    curve["Spot"].values,
    extrapolate=False
)

# ============================================================
# 2. DISCOUNT FUNCTION
# ============================================================

def spot_rate(t):
    """
    Semiannual-compounded spot rate.

    For t < 0.5Y:
        use the 0.5Y spot rate, applied over actual t.

    For 0.5Y <= t <= 30Y:
        PCHIP interpolate the bootstrapped spot curve.
    """
    if t <= 0:
        return 0.0

    if t < 0.5:
        return float(curve["Spot"].iloc[0])

    if t > 30.0:
        raise ValueError(f"Cash flow at {t:.4f}Y exceeds curve horizon.")

    return float(spot_interp(t))


def discount_factor(t):
    s = spot_rate(t)

    # Semiannual compounding convention
    return 1.0 / ((1.0 + s / 2.0) ** (2.0 * t))


# ============================================================
# 3. QUERY TREASURYDIRECT FOR EACH SECURITY
# ============================================================

TD_URL = "https://www.treasurydirect.gov/TA_WS/securities/search"

def get_security(cusip):

    params = {
        "format": "json",
        "cusip": cusip
    }

    r = requests.get(TD_URL, params=params, timeout=30)
    r.raise_for_status()

    data = r.json()

    if not data:
        raise RuntimeError(f"TreasuryDirect returned no data for {cusip}")

    # TreasuryDirect can return auction/reopening records for same CUSIP.
    # All must describe the same underlying security.
    records = data if isinstance(data, list) else [data]

    def first_non_null(keys):
        for rec in records:
            for key in keys:
                if key in rec and rec[key] not in (None, ""):
                    return rec[key]
        return None

    security_type = first_non_null(
        ["securityType", "type"]
    )

    coupon = first_non_null(
        ["interestRate", "interestRatePct", "couponRate"]
    )

    maturity = first_non_null(
        ["maturityDate"]
    )

    issue_date = first_non_null(
        ["issueDate"]
    )

    if coupon is None or maturity is None:
        raise RuntimeError(
            f"Missing coupon/maturity data for {cusip}.\n"
            f"Returned fields: {records[0].keys()}"
        )

    coupon = float(coupon)
    maturity = pd.Timestamp(maturity)
    issue_date = pd.Timestamp(issue_date) if issue_date else pd.NaT

    # Hard exclusions
    type_text = str(security_type).lower()

    if any(x in type_text for x in ["bill", "tips", "frn", "floating"]):
        raise RuntimeError(
            f"{cusip} is not an ordinary nominal fixed-coupon Note/Bond: "
            f"{security_type}"
        )

    return {
        "CUSIP": cusip,
        "Security Type": security_type,
        "Coupon": coupon,
        "Issue Date": issue_date,
        "Maturity": maturity
    }


# ============================================================
# 4. GENERATE ACTUAL SEMIANNUAL COUPON DATES
# ============================================================

def coupon_schedule(maturity, valuation_date):

    maturity = pd.Timestamp(maturity)

    dates = [maturity]
    d = maturity

    # Work backwards in exact six-month calendar intervals
    while True:
        d = d - relativedelta(months=6)

        if d <= valuation_date:
            previous_coupon = d
            break

        dates.append(d)

    dates = sorted(dates)

    next_coupon = dates[0]

    return previous_coupon, next_coupon, dates


# ============================================================
# 5. ACT/ACT ACCRUED INTEREST
# ============================================================

def accrued_interest(coupon_pct, previous_coupon,
                     next_coupon, valuation_date):

    coupon_payment = FACE * (coupon_pct / 100.0) / 2.0

    days_accrued = (valuation_date - previous_coupon).days
    days_period = (next_coupon - previous_coupon).days

    fraction = days_accrued / days_period

    ai = coupon_payment * fraction

    return ai, fraction


# ============================================================
# 6. DCF ONE TREASURY
# ============================================================

def price_treasury(sec):

    coupon_pct = sec["Coupon"]
    maturity = sec["Maturity"]

    coupon_payment = FACE * (coupon_pct / 100.0) / 2.0

    previous_coupon, next_coupon, payment_dates = coupon_schedule(
        maturity,
        VALUATION_DATE
    )

    pv_total = 0.0
    cashflow_rows = []

    for payment_date in payment_dates:

        # ACT/365 time from valuation date for curve horizon
        t = (payment_date - VALUATION_DATE).days / 365.0

        cf = coupon_payment

        if payment_date == maturity:
            cf += FACE

        s = spot_rate(t)
        df = discount_factor(t)
        pv = cf * df

        pv_total += pv

        cashflow_rows.append({
            "CUSIP": sec["CUSIP"],
            "Payment Date": payment_date,
            "Years": t,
            "Cash Flow": cf,
            "Spot Rate": s,
            "Discount Factor": df,
            "PV": pv
        })

    dirty_price = pv_total

    ai, accrued_fraction = accrued_interest(
        coupon_pct,
        previous_coupon,
        next_coupon,
        VALUATION_DATE
    )

    clean_price = dirty_price - ai

    result = {
        "CUSIP": sec["CUSIP"],
        "Type": sec["Security Type"],
        "Coupon (%)": coupon_pct,
        "Maturity": maturity.date(),
        "Previous Coupon": previous_coupon.date(),
        "Next Coupon": next_coupon.date(),
        "Accrued Fraction": accrued_fraction,
        "Accrued Interest": ai,
        "DCF Dirty Price": dirty_price,
        "DCF Clean Price": clean_price
    }

    return result, cashflow_rows


# ============================================================
# 7. QUERY + VALIDATE + PRICE ALL 15
# ============================================================

results = []
all_cashflows = []
securities = []

for cusip in CUSIPS:

    sec = get_security(cusip)
    securities.append(sec)

    result, cashflows = price_treasury(sec)

    results.append(result)
    all_cashflows.extend(cashflows)


security_df = pd.DataFrame(securities)
result_df = pd.DataFrame(results)
cashflow_df = pd.DataFrame(all_cashflows)


# ============================================================
# 8. SANITY CHECKS
# ============================================================

assert len(result_df) == 15
assert result_df["CUSIP"].nunique() == 15

assert (result_df["DCF Dirty Price"] > 0).all()
assert (result_df["DCF Clean Price"] > 0).all()

# Dirty must equal clean + accrued
assert np.allclose(
    result_df["DCF Dirty Price"],
    result_df["DCF Clean Price"] + result_df["Accrued Interest"],
    atol=1e-10
)

# Every CF must occur after valuation date
assert (cashflow_df["Payment Date"] > VALUATION_DATE).all()

# DFs must be positive and <= 1
assert (
    (cashflow_df["Discount Factor"] > 0) &
    (cashflow_df["Discount Factor"] <= 1)
).all()


# ============================================================
# 9. OUTPUT
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("\nVERIFIED TREASURY SECURITY DETAILS")
print("=" * 100)

print(
    security_df[
        ["CUSIP", "Security Type", "Coupon", "Issue Date", "Maturity"]
    ].to_string(index=False)
)

print("\n\nDCF RESULTS — VALUATION DATE 2026-09-17")
print("=" * 100)

print(
    result_df[
        [
            "CUSIP",
            "Coupon (%)",
            "Maturity",
            "Previous Coupon",
            "Next Coupon",
            "Accrued Interest",
            "DCF Dirty Price",
            "DCF Clean Price"
        ]
    ].to_string(index=False)
)


# ============================================================
# 10. SAVE RESULTS
# ============================================================

result_file = "/content/treasury_dcf_results.csv"
cashflow_file = "/content/treasury_dcf_cashflows.csv"
security_file = "/content/treasury_verified_securities.csv"

result_df.to_csv(result_file, index=False)
cashflow_df.to_csv(cashflow_file, index=False)
security_df.to_csv(security_file, index=False)

print("\nSaved:")
print(result_file)
print(cashflow_file)
print(security_file)


VERIFIED TREASURY SECURITY DETAILS
    CUSIP Security Type   Coupon Issue Date   Maturity
91282CLL3          Note 3.375000 2024-09-16 2027-09-15
9128284N7          Note 2.875000 2018-07-16 2028-05-15
912828YS3          Note 1.750000 2020-01-15 2029-11-15
912828Z94          Note 1.500000 2020-04-15 2030-02-15
91282CCB5          Note 1.625000 2021-07-15 2031-05-15
91282CDY4          Note 1.875000 2022-04-18 2032-02-15
91282CGM7          Note 3.500000 2023-04-17 2033-02-15
91282CJY8          Note 1.750000 2024-05-31 2034-01-15
91282CNS6          Note 1.875000 2025-11-28 2035-07-15
912810QF8          Bond 2.125000 2010-08-31 2040-02-15
912810RL4          Bond 0.750000 2015-10-30 2045-02-15
912810SG4          Bond 1.000000 2019-08-30 2049-02-15
912810SM1          Bond 0.250000 2020-08-31 2050-02-15
912810TP3          Bond 1.500000 2023-08-31 2053-02-15
912810UG1          Bond 4.625000 2025-04-15 2055-02-15


DCF RESULTS — VALUATION DATE 2026-09-17
    CUSIP  Coupon (%)   Maturity Previous 

In [ ]:
# ============================================================
# NOMINAL U.S. TREASURY DCF vs IBKR MARKET YTM — CORRECTED
# Valuation date: 17-Sep-2026
# ============================================================

import numpy as np
import pandas as pd
import requests
from scipy.interpolate import PchipInterpolator
from pandas.tseries.offsets import DateOffset, MonthEnd
import plotly.graph_objects as go

VALUATION_DATE = pd.Timestamp("2026-09-17")

# ------------------------------------------------------------
# 1. Securities + observed IBKR YTMs
# ------------------------------------------------------------

securities = pd.DataFrame({
    "CUSIP": [
        "91282CQY0",
        "91282CQR5",
        "91282CQU8",
        "91282CQT1",
        "91282CQQ7",
        "912810UV8",
        "912810UG1",
        "912810UU0",
        "91282CQD6"
    ],
    "Market_YTM_pct": [
        4.680,
        4.740,
        4.810,
        4.870,
        4.955,
        5.350,
        5.380,
        5.320,
        4.800
    ]
})

# ------------------------------------------------------------
# 2. Load bootstrapped spot curve
# ------------------------------------------------------------

curve = pd.read_csv("/content/treasury_bootstrapped_spot_curve.csv")

maturity_col = next(
    c for c in curve.columns
    if "matur" in c.lower() or "year" in c.lower()
)

spot_col = next(
    c for c in curve.columns
    if "spot" in c.lower()
)

curve[maturity_col] = pd.to_numeric(curve[maturity_col])
curve[spot_col] = pd.to_numeric(curve[spot_col])

spot_values = curve[spot_col].values.astype(float)

if np.nanmedian(spot_values) > 1:
    spot_values /= 100.0

spot_interp = PchipInterpolator(
    curve[maturity_col].values,
    spot_values,
    extrapolate=False
)

six_month_spot = float(spot_interp(0.5))


def spot_rate(t):
    if t < 0.5:
        return six_month_spot

    s = float(spot_interp(t))

    if not np.isfinite(s):
        raise ValueError(f"No spot rate for t={t:.6f}")

    return s


# ------------------------------------------------------------
# 3. TreasuryDirect metadata
# ------------------------------------------------------------

def get_security(cusip):

    url = "https://www.treasurydirect.gov/TA_WS/securities/search"

    r = requests.get(
        url,
        params={"format": "json", "cusip": cusip},
        timeout=30
    )

    r.raise_for_status()
    data = r.json()

    if not data:
        raise ValueError(f"No TreasuryDirect security found: {cusip}")

    x = data[0]

    return {
        "CUSIP": cusip,
        "Security_Type": x.get("securityType"),
        "Coupon_pct": float(x["interestRate"]),
        "Issue_Date": pd.Timestamp(x["issueDate"]).normalize(),
        "Maturity": pd.Timestamp(x["maturityDate"]).normalize()
    }


metadata = pd.DataFrame(
    [get_security(c) for c in securities["CUSIP"]]
)

df = securities.merge(metadata, on="CUSIP", how="left")


# ------------------------------------------------------------
# 4. ROBUST SEMIANNUAL COUPON SCHEDULE
# ------------------------------------------------------------

def shift_months_preserve_eom(date, months):
    """
    Shift by months while preserving end-of-month status.

    Example:
    31-May -> 30-Nov -> 31-May
    rather than allowing February/calendar drift.
    """
    date = pd.Timestamp(date)

    is_eom = date == date + MonthEnd(0)

    shifted = date + DateOffset(months=months)

    if is_eom:
        shifted = shifted + MonthEnd(0)

    return pd.Timestamp(shifted).normalize()


def coupon_schedule(maturity, valuation_date):

    maturity = pd.Timestamp(maturity).normalize()
    valuation_date = pd.Timestamp(valuation_date).normalize()

    # Build ALL dates backwards from maturity.
    dates = [maturity]

    d = maturity

    while d > valuation_date - DateOffset(years=1):
        d = shift_months_preserve_eom(d, -6)
        dates.append(d)

    dates = sorted(set(dates))

    previous = [d for d in dates if d <= valuation_date]
    following = [d for d in dates if d > valuation_date]

    if not previous or not following:
        raise ValueError(
            f"Could not establish coupon period for maturity {maturity}"
        )

    prev_coupon = max(previous)
    next_coupon = min(following)

    # Generate future coupons FORWARD from next coupon,
    # preserving the maturity's coupon convention.
    future_dates = []

    d = next_coupon

    while d < maturity:
        future_dates.append(d)
        d = shift_months_preserve_eom(d, 6)

    # Principal + final coupon
    if maturity not in future_dates:
        future_dates.append(maturity)

    future_dates = sorted(set(future_dates))

    return prev_coupon, next_coupon, future_dates


# ------------------------------------------------------------
# 5. Price securities
# ------------------------------------------------------------

results = []

for _, row in df.iterrows():

    cusip = row["CUSIP"]
    maturity = row["Maturity"]

    coupon_rate = row["Coupon_pct"] / 100
    ytm = row["Market_YTM_pct"] / 100

    coupon_payment = 100 * coupon_rate / 2

    prev_coupon, next_coupon, future_dates = coupon_schedule(
        maturity,
        VALUATION_DATE
    )

    # ---------------------------
    # Accrued interest
    # ---------------------------

    accrued_days = (VALUATION_DATE - prev_coupon).days
    period_days = (next_coupon - prev_coupon).days

    accrued_interest = (
        coupon_payment *
        accrued_days /
        period_days
    )

    # ---------------------------
    # DCF theoretical price
    # ---------------------------

    dcf_dirty = 0.0

    for payment_date in future_dates:

        t = (payment_date - VALUATION_DATE).days / 365.0

        s = spot_rate(t)

        discount_factor = (
            1 / (1 + s / 2) ** (2 * t)
        )

        cashflow = coupon_payment

        if payment_date == maturity:
            cashflow += 100

        dcf_dirty += cashflow * discount_factor

    dcf_clean = dcf_dirty - accrued_interest

    # ---------------------------
    # Market price from IBKR YTM
    # ---------------------------

    days_to_next = (next_coupon - VALUATION_DATE).days
    days_in_period = (next_coupon - prev_coupon).days

    fraction_to_next = (
        days_to_next / days_in_period
    )

    market_dirty = 0.0

    for i, payment_date in enumerate(future_dates):

        periods = fraction_to_next + i

        cashflow = coupon_payment

        if payment_date == maturity:
            cashflow += 100

        market_dirty += (
            cashflow /
            (1 + ytm / 2) ** periods
        )

    market_clean = market_dirty - accrued_interest

    # ---------------------------
    # Relative value
    # ---------------------------

    difference = dcf_clean - market_clean

    mispricing_pct = (
        difference / market_clean * 100
    )

    if mispricing_pct > 0:
        signal = "Undervalued"
    elif mispricing_pct < 0:
        signal = "Overvalued"
    else:
        signal = "Fair Value"

    years_to_maturity = (
        maturity - VALUATION_DATE
    ).days / 365.0

    results.append({
        "CUSIP": cusip,
        "Coupon_pct": row["Coupon_pct"],
        "Maturity": maturity.date(),
        "Years_to_Maturity": years_to_maturity,
        "Market_YTM_pct": row["Market_YTM_pct"],
        "Prev_Coupon": prev_coupon.date(),
        "Next_Coupon": next_coupon.date(),
        "Accrued_Interest": accrued_interest,
        "Market_Clean": market_clean,
        "DCF_Clean": dcf_clean,
        "Price_Difference": difference,
        "Mispricing_pct": mispricing_pct,
        "Signal": signal
    })


results = (
    pd.DataFrame(results)
    .sort_values("Years_to_Maturity")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. HARD SANITY CHECKS
# ------------------------------------------------------------

assert len(results) == 9
assert results["CUSIP"].is_unique
assert (results["Market_Clean"] > 70).all(), \
    "Implausibly low Treasury market price detected."
assert (results["DCF_Clean"] > 70).all(), \
    "Implausibly low Treasury DCF price detected."
assert np.isfinite(results["Mispricing_pct"]).all()

print("\nDCF vs MARKET")
print("=" * 125)

display_cols = [
    "CUSIP",
    "Coupon_pct",
    "Maturity",
    "Market_YTM_pct",
    "Market_Clean",
    "DCF_Clean",
    "Price_Difference",
    "Mispricing_pct",
    "Signal"
]

out = results[display_cols].copy()

for c in [
    "Coupon_pct",
    "Market_YTM_pct",
    "Market_Clean",
    "DCF_Clean",
    "Price_Difference",
    "Mispricing_pct"
]:
    out[c] = out[c].round(4)

print(out.to_string(index=False))


# ------------------------------------------------------------
# 7. CHART — DCF vs Market
# ------------------------------------------------------------

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=results["Years_to_Maturity"],
    y=results["Market_Clean"],
    mode="lines+markers",
    name="Market Clean Price",
    text=results["CUSIP"],
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Market: %{y:.4f}"
        "<extra></extra>"
    )
))

fig1.add_trace(go.Scatter(
    x=results["Years_to_Maturity"],
    y=results["DCF_Clean"],
    mode="lines+markers",
    name="DCF Clean Price",
    text=results["CUSIP"],
    hovertemplate=(
        "<b>%{text}</b><br>"
        "DCF: %{y:.4f}"
        "<extra></extra>"
    )
))

fig1.update_layout(
    title="Nominal U.S. Treasuries — DCF vs Market Clean Price",
    xaxis_title="Years to Maturity",
    yaxis_title="Clean Price ($ per $100 Face)",
    template="plotly_white",
    hovermode="x unified",
    width=1050,
    height=600
)

fig1.show()


# ------------------------------------------------------------
# 8. CHART — Mispricing %
# ------------------------------------------------------------

colors = [
    "green" if x > 0 else "red"
    for x in results["Mispricing_pct"]
]

fig2 = go.Figure(go.Bar(
    x=results["CUSIP"],
    y=results["Mispricing_pct"],
    marker_color=colors,
    text=[
        f"{x:+.3f}%"
        for x in results["Mispricing_pct"]
    ],
    textposition="outside"
))

fig2.add_hline(
    y=0,
    line_color="black",
    line_width=1
)

fig2.update_layout(
    title="Relative Value vs Bootstrapped Treasury Spot Curve",
    xaxis_title="CUSIP",
    yaxis_title="DCF Premium / Discount to Market (%)",
    template="plotly_white",
    width=1050,
    height=600
)

fig2.show()


# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

results.to_csv(
    "/content/treasury_nominal_dcf_vs_market.csv",
    index=False
)

print("\nSaved: /content/treasury_nominal_dcf_vs_market.csv")


DCF vs MARKET
    CUSIP  Coupon_pct   Maturity  Market_YTM_pct  Market_Clean  DCF_Clean  Price_Difference  Mispricing_pct      Signal
91282CQY0    4.125000 2028-06-30        4.680000     99.054100  99.131300          0.077200        0.078000 Undervalued
91282CQR5    3.875000 2029-05-15        4.740000     97.856700  97.870000          0.013300        0.013500 Undervalued
91282CQD6    3.500000 2031-02-28        4.800000     94.841400  94.948800          0.107400        0.113300 Undervalued
91282CQU8    4.125000 2031-05-31        4.810000     97.141800  97.274700          0.132900        0.136800 Undervalued
91282CQT1    4.250000 2033-05-31        4.870000     96.484500  96.573200          0.088700        0.091900 Undervalued
91282CQQ7    4.375000 2036-05-15        4.955000     95.584000  95.728800          0.144800        0.151500 Undervalued
912810UV8    5.000000 2046-05-15        5.350000     95.767700  96.032800          0.265100        0.276800 Undervalued
912810UG1    4.625000 205


Saved: /content/treasury_nominal_dcf_vs_market.csv


In [ ]:
# ============================================================
# NOMINAL U.S. TREASURY INTEREST-RATE RISK
#
# Calculates:
#   1. Macaulay Duration
#   2. Modified Duration
#   3. DV01 per $100 face
#   4. Analytical Convexity
#   5. Numerical ±1 bp validation
#
# Requires successful DCF cell above:
#   - results
#   - coupon_schedule()
#   - VALUATION_DATE
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# ------------------------------------------------------------
# 1. BOND RISK FUNCTION
# ------------------------------------------------------------

def calculate_bond_risk(coupon_pct, maturity, ytm_pct):

    maturity = pd.Timestamp(maturity)
    valuation_date = pd.Timestamp(VALUATION_DATE)

    coupon_rate = coupon_pct / 100.0
    y = ytm_pct / 100.0

    # Semiannual Treasury coupon
    coupon_payment = 100 * coupon_rate / 2

    # Existing verified coupon schedule function
    prev_coupon, next_coupon, future_dates = coupon_schedule(
        maturity,
        valuation_date
    )

    # --------------------------------------------------------
    # Fractional coupon period
    # --------------------------------------------------------

    days_to_next = (next_coupon - valuation_date).days
    days_in_period = (next_coupon - prev_coupon).days

    fraction_to_next = days_to_next / days_in_period

    # Accrued interest
    accrued_days = (valuation_date - prev_coupon).days

    accrued_interest = (
        coupon_payment *
        accrued_days /
        days_in_period
    )

    # --------------------------------------------------------
    # Build future cash flows
    # --------------------------------------------------------

    cashflows = []

    for i, payment_date in enumerate(future_dates):

        # Number of semiannual periods from settlement
        n = fraction_to_next + i

        # Equivalent time in years
        t = n / 2.0

        cf = coupon_payment

        if payment_date == maturity:
            cf += 100

        # PV using observed market YTM
        pv = (
            cf /
            (1 + y / 2) ** n
        )

        cashflows.append({
            "Payment_Date": payment_date,
            "Periods": n,
            "Time_Years": t,
            "Cash_Flow": cf,
            "PV": pv
        })

    cf = pd.DataFrame(cashflows)

    # --------------------------------------------------------
    # Market dirty and clean prices
    # --------------------------------------------------------

    dirty_price = cf["PV"].sum()

    clean_price = (
        dirty_price -
        accrued_interest
    )

    # ========================================================
    # 2. MACAULAY DURATION
    # ========================================================

    macaulay_duration = (
        (
            cf["Time_Years"] *
            cf["PV"]
        ).sum()
        /
        dirty_price
    )

    # ========================================================
    # 3. MODIFIED DURATION
    # ========================================================

    modified_duration = (
        macaulay_duration /
        (1 + y / 2)
    )

    # ========================================================
    # 4. DV01
    #
    # Approximate dollar price change for a 1 bp parallel
    # change in YTM, per $100 face value.
    # ========================================================

    dv01 = (
        modified_duration *
        dirty_price *
        0.0001
    )

    # ========================================================
    # 5. ANALYTICAL CONVEXITY
    #
    # For semiannual-compounded YTM:
    #
    # C = [1/P * sum(CF*n*(n+1) /
    #     (1+y/2)^(n+2))] / 4
    #
    # Division by 4 converts semiannual-yield sensitivity
    # into annual-yield units.
    # ========================================================

    convexity_numerator = np.sum(
        cf["Cash_Flow"] *
        cf["Periods"] *
        (cf["Periods"] + 1) /
        (1 + y / 2) **
        (cf["Periods"] + 2)
    )

    convexity = (
        convexity_numerator /
        dirty_price /
        4.0
    )

    # ========================================================
    # 6. NUMERICAL ±1 BP VALIDATION
    # ========================================================

    bump = 0.0001  # 1 basis point

    def dirty_price_at_yield(y_new):

        return np.sum(
            cf["Cash_Flow"] /
            (1 + y_new / 2) **
            cf["Periods"]
        )

    P0 = dirty_price

    # Yield DOWN 1 bp
    P_down = dirty_price_at_yield(
        y - bump
    )

    # Yield UP 1 bp
    P_up = dirty_price_at_yield(
        y + bump
    )

    # Central-difference modified duration
    numerical_duration = (
        P_down - P_up
    ) / (
        2 *
        P0 *
        bump
    )

    # Central-difference convexity
    numerical_convexity = (
        P_down +
        P_up -
        2 * P0
    ) / (
        P0 *
        bump**2
    )

    return {

        "Market_Dirty":
            dirty_price,

        "Market_Clean_Check":
            clean_price,

        "Macaulay_Duration":
            macaulay_duration,

        "Modified_Duration":
            modified_duration,

        "DV01_per_100":
            dv01,

        "Convexity":
            convexity,

        "Numerical_ModDur":
            numerical_duration,

        "Numerical_Convexity":
            numerical_convexity
    }


# ------------------------------------------------------------
# 7. CALCULATE ALL NINE NOMINAL TREASURIES
# ------------------------------------------------------------

risk_rows = []

for _, row in results.iterrows():

    metrics = calculate_bond_risk(

        coupon_pct=
            row["Coupon_pct"],

        maturity=
            row["Maturity"],

        ytm_pct=
            row["Market_YTM_pct"]
    )

    risk_rows.append({

        "CUSIP":
            row["CUSIP"],

        "Maturity":
            row["Maturity"],

        "Years_to_Maturity":
            row["Years_to_Maturity"],

        "Market_YTM_pct":
            row["Market_YTM_pct"],

        **metrics
    })


risk = pd.DataFrame(risk_rows)

risk = (
    risk
    .sort_values("Years_to_Maturity")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. CREATE VALIDATION ERROR COLUMNS
# ------------------------------------------------------------

risk["Duration_Error"] = (
    risk["Modified_Duration"] -
    risk["Numerical_ModDur"]
)

risk["Convexity_Error"] = (
    risk["Convexity"] -
    risk["Numerical_Convexity"]
)


# ------------------------------------------------------------
# 9. VALIDATE MARKET PRICES AGAINST DCF CELL
# ------------------------------------------------------------

previous_prices = (
    results
    .set_index("CUSIP")
    ["Market_Clean"]
)

for _, row in risk.iterrows():

    original_price = (
        previous_prices.loc[
            row["CUSIP"]
        ]
    )

    assert abs(
        row["Market_Clean_Check"] -
        original_price
    ) < 1e-8, (
        f"Market-price reproduction failed "
        f"for {row['CUSIP']}"
    )


# ------------------------------------------------------------
# 10. VALIDATE ANALYTICAL RISK MEASURES
#
# Slight tolerance required because numerical calculation
# uses finite ±1 bp movements.
# ------------------------------------------------------------

max_duration_error = (
    risk["Duration_Error"]
    .abs()
    .max()
)

max_convexity_error = (
    risk["Convexity_Error"]
    .abs()
    .max()
)

assert (
    max_duration_error < 2e-5
), (
    f"Duration validation failed. "
    f"Max error = {max_duration_error}"
)

assert (
    max_convexity_error < 1e-3
), (
    f"Convexity validation failed. "
    f"Max error = {max_convexity_error}"
)


print("✓ Market-price reproduction passed")
print("✓ Analytical duration validation passed")
print("✓ Analytical convexity validation passed")

print(
    f"\nMaximum duration validation error: "
    f"{max_duration_error:.10f}"
)

print(
    f"Maximum convexity validation error: "
    f"{max_convexity_error:.10f}"
)


# ------------------------------------------------------------
# 11. FINAL RISK TABLE
# ------------------------------------------------------------

display_cols = [

    "CUSIP",
    "Maturity",
    "Market_YTM_pct",
    "Market_Clean_Check",
    "Macaulay_Duration",
    "Modified_Duration",
    "DV01_per_100",
    "Convexity"
]

output = risk[
    display_cols
].copy()

round_cols = [

    "Market_YTM_pct",
    "Market_Clean_Check",
    "Macaulay_Duration",
    "Modified_Duration",
    "DV01_per_100",
    "Convexity"
]

for col in round_cols:
    output[col] = (
        output[col]
        .round(4)
    )


print(
    "\nTREASURY INTEREST-RATE RISK"
)

print("=" * 125)

print(
    output.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 12. ANALYTICAL vs NUMERICAL VALIDATION TABLE
# ------------------------------------------------------------

validation = risk[[
    "CUSIP",
    "Modified_Duration",
    "Numerical_ModDur",
    "Duration_Error",
    "Convexity",
    "Numerical_Convexity",
    "Convexity_Error"
]].copy()

print(
    "\nANALYTICAL vs NUMERICAL VALIDATION"
)

print("=" * 125)

print(
    validation
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. PLOT — MODIFIED DURATION
# ------------------------------------------------------------

fig1 = go.Figure()

fig1.add_trace(

    go.Bar(

        x=
            risk["CUSIP"],

        y=
            risk["Modified_Duration"],

        text=[
            f"{x:.2f}"
            for x in
            risk["Modified_Duration"]
        ],

        textposition=
            "outside",

        customdata=
            risk[[
                "Market_YTM_pct",
                "DV01_per_100",
                "Convexity"
            ]],

        hovertemplate=(

            "<b>%{x}</b><br>"

            "Modified Duration: "
            "%{y:.3f} years<br>"

            "YTM: "
            "%{customdata[0]:.3f}%<br>"

            "DV01/$100: "
            "%{customdata[1]:.4f}<br>"

            "Convexity: "
            "%{customdata[2]:.2f}"

            "<extra></extra>"
        )
    )
)

fig1.update_layout(

    title=
        "Modified Duration — Nominal U.S. Treasuries",

    xaxis_title=
        "CUSIP",

    yaxis_title=
        "Modified Duration (Years)",

    template=
        "plotly_white",

    width=
        1050,

    height=
        600
)

fig1.show()


# ------------------------------------------------------------
# 14. PLOT — CONVEXITY
# ------------------------------------------------------------

fig2 = go.Figure()

fig2.add_trace(

    go.Bar(

        x=
            risk["CUSIP"],

        y=
            risk["Convexity"],

        text=[
            f"{x:.1f}"
            for x in
            risk["Convexity"]
        ],

        textposition=
            "outside",

        hovertemplate=(

            "<b>%{x}</b><br>"

            "Convexity: "
            "%{y:.3f}"

            "<extra></extra>"
        )
    )
)

fig2.update_layout(

    title=
        "Convexity — Nominal U.S. Treasuries",

    xaxis_title=
        "CUSIP",

    yaxis_title=
        "Convexity",

    template=
        "plotly_white",

    width=
        1050,

    height=
        600
)

fig2.show()


# ------------------------------------------------------------
# 15. SAVE RESULTS
# ------------------------------------------------------------

output_path = (
    "/content/"
    "treasury_duration_convexity.csv"
)

risk.to_csv(
    output_path,
    index=False
)

print(
    f"\nSaved: {output_path}"
)

✓ Market-price reproduction passed
✓ Analytical duration validation passed
✓ Analytical convexity validation passed

Maximum duration validation error: 0.0000142251
Maximum convexity validation error: 0.0001972827

TREASURY INTEREST-RATE RISK
    CUSIP   Maturity  Market_YTM_pct  Market_Clean_Check  Macaulay_Duration  Modified_Duration  DV01_per_100  Convexity
91282CQY0 2028-06-30        4.680000           99.054100           1.725200           1.685700      0.016800   3.729100
91282CQR5 2029-05-15        4.740000           97.856700           2.519300           2.461000      0.024400   7.488600
91282CQD6 2031-02-28        4.800000           94.841400           4.145700           4.048500      0.038500  19.121000
91282CQU8 2031-05-31        4.810000           97.141800           4.263000           4.162900      0.041000  20.528600
91282CQT1 2033-05-31        4.870000           96.484500           5.810000           5.671900      0.055400  38.106200
91282CQQ7 2036-05-15        4.955000 


Saved: /content/treasury_duration_convexity.csv


In [ ]:
# ============================================================
# TASK 1 — NOMINAL TREASURY DCF + RISK SUMMARY
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display


# ------------------------------------------------------------
# 1. CHECK REQUIRED DATA EXISTS
# ------------------------------------------------------------

required_results = [
    "CUSIP",
    "Coupon_pct",
    "Maturity",
    "Market_YTM_pct",
    "Market_Clean",
    "DCF_Clean",
    "Mispricing_pct"
]

required_risk = [
    "CUSIP",
    "Macaulay_Duration",
    "Modified_Duration",
    "DV01_per_100",
    "Convexity"
]

missing_results = [
    c for c in required_results
    if c not in results.columns
]

missing_risk = [
    c for c in required_risk
    if c not in risk.columns
]

if missing_results:
    raise ValueError(
        f"Missing columns from results: {missing_results}\n"
        f"Available columns: {list(results.columns)}"
    )

if missing_risk:
    raise ValueError(
        f"Missing columns from risk: {missing_risk}\n"
        f"Available columns: {list(risk.columns)}"
    )


# ------------------------------------------------------------
# 2. MERGE DCF VALUATION + RISK RESULTS
# ------------------------------------------------------------

summary = pd.merge(
    results[required_results],
    risk[required_risk],
    on="CUSIP",
    how="inner",
    validate="one_to_one"
)

summary["Maturity"] = pd.to_datetime(summary["Maturity"])

summary = (
    summary
    .sort_values("Maturity")
    .reset_index(drop=True)
)

summary["DCF_minus_Market"] = (
    summary["DCF_Clean"] -
    summary["Market_Clean"]
)


# ------------------------------------------------------------
# 3. SANITY CHECK
# ------------------------------------------------------------

assert len(summary) == len(results), (
    f"Merge lost securities: results={len(results)}, "
    f"merged={len(summary)}"
)

print(f"✓ Successfully merged {len(summary)} nominal Treasuries")


# ------------------------------------------------------------
# 4. CREATE REPORT TABLE
# ------------------------------------------------------------

report_table = summary[[
    "CUSIP",
    "Maturity",
    "Coupon_pct",
    "Market_YTM_pct",
    "Market_Clean",
    "DCF_Clean",
    "DCF_minus_Market",
    "Mispricing_pct",
    "Macaulay_Duration",
    "Modified_Duration",
    "DV01_per_100",
    "Convexity"
]].copy()

report_table.columns = [
    "CUSIP",
    "Maturity",
    "Coupon (%)",
    "YTM (%)",
    "Market Clean",
    "DCF Clean",
    "DCF − Market",
    "Mispricing (%)",
    "Macaulay Duration",
    "Modified Duration",
    "DV01 / $100",
    "Convexity"
]

report_table["Maturity"] = (
    report_table["Maturity"]
    .dt.strftime("%d-%b-%Y")
)


# ------------------------------------------------------------
# 5. DISPLAY FORMATTED TABLE
# ------------------------------------------------------------

print("\n")
print("=" * 150)
print("NOMINAL U.S. TREASURY — DCF VALUATION AND INTEREST-RATE RISK")
print("=" * 150)

styled_table = (
    report_table.style
    .format({
        "Coupon (%)": "{:.3f}",
        "YTM (%)": "{:.3f}",
        "Market Clean": "{:.3f}",
        "DCF Clean": "{:.3f}",
        "DCF − Market": "{:.3f}",
        "Mispricing (%)": "{:.3f}",
        "Macaulay Duration": "{:.3f}",
        "Modified Duration": "{:.3f}",
        "DV01 / $100": "{:.4f}",
        "Convexity": "{:.2f}"
    })
    .set_properties(**{
        "text-align": "center"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-weight", "bold")
            ]
        }
    ])
)

display(styled_table)


# ------------------------------------------------------------
# 6. KEY EMPIRICAL FINDINGS
# ------------------------------------------------------------

largest_mispricing = summary.loc[
    summary["Mispricing_pct"].abs().idxmax()
]

highest_duration = summary.loc[
    summary["Modified_Duration"].idxmax()
]

highest_dv01 = summary.loc[
    summary["DV01_per_100"].idxmax()
]

highest_convexity = summary.loc[
    summary["Convexity"].idxmax()
]

print("\nKEY FINDINGS")
print("=" * 90)

print(
    f"Largest absolute DCF/market discrepancy: "
    f"{largest_mispricing['CUSIP']} | "
    f"{largest_mispricing['Mispricing_pct']:.3f}%"
)

print(
    f"Mean absolute DCF/market discrepancy: "
    f"{summary['Mispricing_pct'].abs().mean():.3f}%"
)

print(
    f"Median absolute DCF/market discrepancy: "
    f"{summary['Mispricing_pct'].abs().median():.3f}%"
)

print(
    f"Highest modified duration: "
    f"{highest_duration['CUSIP']} | "
    f"{highest_duration['Modified_Duration']:.3f} years"
)

print(
    f"Highest DV01: "
    f"{highest_dv01['CUSIP']} | "
    f"${highest_dv01['DV01_per_100']:.4f} per $100 face"
)

print(
    f"Highest convexity: "
    f"{highest_convexity['CUSIP']} | "
    f"{highest_convexity['Convexity']:.2f}"
)


# ------------------------------------------------------------
# 7. MODEL VALIDATION
# ------------------------------------------------------------

max_duration_error = (
    risk["Modified_Duration"] -
    risk["Numerical_ModDur"]
).abs().max()

max_convexity_error = (
    risk["Convexity"] -
    risk["Numerical_Convexity"]
).abs().max()

print("\nMODEL VALIDATION")
print("=" * 90)

print("Market-price reproduction: PASSED")

print(
    f"Duration analytical vs numerical: PASSED "
    f"(max error = {max_duration_error:.8f})"
)

print(
    f"Convexity analytical vs numerical: PASSED "
    f"(max error = {max_convexity_error:.8f})"
)


# ------------------------------------------------------------
# 8. SAVE REPORT-READY DATA
# ------------------------------------------------------------

output_path = "/content/task1_nominal_treasury_dcf_risk_summary.csv"

report_table.to_csv(
    output_path,
    index=False
)

print(f"\nSaved: {output_path}")

✓ Successfully merged 9 nominal Treasuries


NOMINAL U.S. TREASURY — DCF VALUATION AND INTEREST-RATE RISK


,CUSIP,Maturity,Coupon (%),YTM (%),Market Clean,DCF Clean,DCF − Market,Mispricing (%),Macaulay Duration,Modified Duration,DV01 / $100,Convexity
0,91282CQY0,30-Jun-2028,4.125,4.680,99.054,99.131,0.077,0.078,1.725,1.686,0.0168,3.73
1,91282CQR5,15-May-2029,3.875,4.740,97.857,97.870,0.013,0.014,2.519,2.461,0.0244,7.49
2,91282CQD6,28-Feb-2031,3.500,4.800,94.841,94.949,0.107,0.113,4.146,4.049,0.0385,19.12
3,91282CQU8,31-May-2031,4.125,4.810,97.142,97.275,0.133,0.137,4.263,4.163,0.0410,20.53
4,91282CQT1,31-May-2033,4.250,4.870,96.484,96.573,0.089,0.092,5.810,5.672,0.0554,38.11
5,91282CQQ7,15-May-2036,4.375,4.955,95.584,95.729,0.145,0.151,7.815,7.626,0.0740,70.49
6,912810UV8,15-May-2046,5.000,5.350,95.768,96.033,0.265,0.277,12.354,12.032,0.1173,198.57
7,912810UG1,15-Feb-2055,4.625,5.380,89.068,89.982,0.915,1.027,15.266,14.867,0.1330,325.31
8,912810UU0,15-May-2056,5.000,5.320,95.245,95.561,0.316,0.332,15.155,14.763,0.1431,329.96



KEY FINDINGS
Largest absolute DCF/market discrepancy: 912810UG1 | 1.027%
Mean absolute DCF/market discrepancy: 0.247%
Median absolute DCF/market discrepancy: 0.137%
Highest modified duration: 912810UG1 | 14.867 years
Highest DV01: 912810UU0 | $0.1431 per $100 face
Highest convexity: 912810UU0 | 329.96

MODEL VALIDATION
Market-price reproduction: PASSED
Duration analytical vs numerical: PASSED (max error = 0.00001423)
Convexity analytical vs numerical: PASSED (max error = 0.00019728)

Saved: /content/task1_nominal_treasury_dcf_risk_summary.csv
